# SCA AutoFix Pipeline

A simple, deterministic pipeline for fixing vulnerable library usages in Java repositories.

**Designed to run on Databricks** (with local fallback for testing).

**Steps:**
1. 📥 Read vulnerable repos from Unity Catalog
2. 📦 Clone the repository
3. 🔍 Get API diff between library versions (clone from GitHub)
4. 🔎 Find library usages in source code
5. 🎯 Match breaking changes with actual usages
6. 🤖 Generate code fix with LLM (**Databricks Foundation Model API**)
7. 💾 Write results back to Unity Catalog

---
## Environment Setup

In [ ]:
# Environment Detection & Configuration
import os
import sys

# ============================================================================
# DATA MODE CONFIGURATION
# ============================================================================
# Set DATA_MODE before anything else
# "mock" = Use sample_data/vulnerable_repos.csv (for local testing)
# "production" = Use Unity Catalog tables (for Databricks)

DATA_MODE = os.environ.get("DATA_MODE", "mock")  # Change to "production" for UC
os.environ["DATA_MODE"] = DATA_MODE  # Ensure it's set for services

# ============================================================================
# Detect if running on Databricks
# ============================================================================
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Auto-switch to production if on Databricks (unless explicitly set to mock)
if ON_DATABRICKS and os.environ.get("DATA_MODE") != "mock":
    DATA_MODE = "production"
    os.environ["DATA_MODE"] = "production"

if ON_DATABRICKS:
    print("🔷 Running on Databricks")
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
else:
    print("💻 Running locally")
    spark = None

# Add the agent package to path
sys.path.insert(0, 'sca-autofix-agent')

# ============================================================================
# Unity Catalog Configuration
# ============================================================================
UC_CATALOG = os.environ.get("UC_CATALOG", "main")
UC_SCHEMA = os.environ.get("UC_SCHEMA", "sca_autofix")

# Local fallback paths
CSV_PATH = "sample_data/vulnerable_repos.csv"
CLONE_DIR = "/tmp/sca-repos"

# ============================================================================
# LLM Configuration (Databricks Foundation Model API)
# ============================================================================
LLM_ENDPOINT = os.environ.get("LLM_ENDPOINT", "databricks-meta-llama-3-1-70b-instruct")
LLM_MAX_TOKENS = int(os.environ.get("LLM_MAX_TOKENS", "4096"))

# ============================================================================
# Library GitHub URLs for API diff comparison
# ============================================================================
LIBRARY_GITHUB_URLS = {
    "jackson-databind": "https://github.com/FasterXML/jackson-databind.git",
    "log4j-core": "https://github.com/apache/logging-log4j2.git",
    "spring-boot": "https://github.com/spring-projects/spring-boot.git",
    "snakeyaml": "https://github.com/snakeyaml/snakeyaml.git",
    "commons-collections": "https://github.com/apache/commons-collections.git",
    "commons-text": "https://github.com/apache/commons-text.git",
    "hibernate-core": "https://github.com/hibernate/hibernate-orm.git",
    "spring-security-core": "https://github.com/spring-projects/spring-security.git",
    "spring-security": "https://github.com/spring-projects/spring-security.git",
    "tomcat-embed-core": "https://github.com/apache/tomcat.git",
    "jjwt-api": "https://github.com/jwtk/jjwt.git",
    "xstream": "https://github.com/x-stream/xstream.git",
    "gson": "https://github.com/google/gson.git",
    "netty-handler": "https://github.com/netty/netty.git",
    "netty": "https://github.com/netty/netty.git",
}

# ============================================================================
# Print configuration summary
# ============================================================================
print(f"\n✅ Configuration loaded")
print(f"   DATA_MODE: {DATA_MODE} {'(auto-switched)' if ON_DATABRICKS else ''}")
if DATA_MODE == "production":
    print(f"   📊 UC: {UC_CATALOG}.{UC_SCHEMA}")
else:
    print(f"   📄 CSV: {CSV_PATH}")
print(f"   CLONE_DIR: {CLONE_DIR}")
print(f"   LLM_ENDPOINT: {LLM_ENDPOINT}")
print(f"   Libraries configured: {len(LIBRARY_GITHUB_URLS)}")

In [ ]:
# Imports
import pandas as pd
import subprocess
import shutil
from pathlib import Path
from typing import List, Dict, Any, Optional
import json
import re

# Databricks-specific imports
if ON_DATABRICKS:
    from databricks.sdk import WorkspaceClient
    from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
    w = WorkspaceClient()
    print("✅ Databricks SDK initialized")
else:
    w = None
    print("✅ Standard imports loaded (no Databricks SDK)")

In [ ]:
# Optional: Import custom services (for advanced usage)
try:
    from services.library_migration_analyzer import LibraryMigrationAnalyzer
    from services.java_analyzer import JavaCodeTransformer
    CUSTOM_SERVICES_AVAILABLE = True
    print("✅ Custom services available")
    print("   - LibraryMigrationAnalyzer: Advanced API diff analysis")
    print("   - JavaCodeTransformer: Java AST parsing")
except ImportError as e:
    CUSTOM_SERVICES_AVAILABLE = False
    print(f"ℹ️ Custom services not loaded: {e}")
    print("   Using built-in functions instead")

---
## Step 1: Read Vulnerable Repos from UC/CSV

In [ ]:
def read_vulnerable_repos(
    status_filter: str = "ready",
    priority_filter: int = None,
    limit: int = 100
) -> pd.DataFrame:
    """
    Read vulnerable repositories from Unity Catalog or CSV file.
    
    Uses UCIntegration service which auto-detects DATA_MODE:
    - "mock" → reads from sample_data/vulnerable_repos.csv
    - "production" → reads from Unity Catalog table
    
    Returns DataFrame with columns:
    - repo_name, repo_url, library, current_version, candidate_versions, cve_id, cvss_score, priority, status
    """
    print("\n" + "="*60)
    print("📥 STEP 1: Reading Vulnerable Repos")
    print("="*60)
    
    try:
        # Use UCIntegration service (supports both mock and production)
        from services.uc_integration import UCIntegration
        
        uc = UCIntegration(catalog=UC_CATALOG, schema=UC_SCHEMA)
        repos = uc.get_vulnerable_repos(
            status=status_filter,
            priority=priority_filter,
            limit=limit
        )
        
        df = pd.DataFrame(repos)
        source = "Unity Catalog" if DATA_MODE == "production" else "CSV (mock)"
        print(f"   Source: {source}")
        
    except ImportError:
        # Fallback: read directly from CSV
        print(f"   Source: CSV file (direct) - {CSV_PATH}")
        df = pd.read_csv(CSV_PATH)
        
        # Parse JSON columns
        import json
        if 'cve_ids' in df.columns:
            df['cve_ids'] = df['cve_ids'].apply(
                lambda x: json.loads(x) if isinstance(x, str) else x
            )
        if 'candidate_versions' in df.columns:
            df['candidate_versions'] = df['candidate_versions'].apply(
                lambda x: json.loads(x) if isinstance(x, str) else x
            )
        
        # Apply filters
        if status_filter:
            df = df[df['status'] == status_filter]
        df = df.head(limit)
    
    print(f"\n   📊 Loaded {len(df)} vulnerable repo/library combinations")
    if len(df) > 0:
        print(f"   📁 Unique repos: {df['repo_name'].nunique()}")
        print(f"   📚 Unique libraries: {df['library'].nunique()}")
        
        # Show summary by priority
        print("\n   Priority breakdown:")
        for priority in ['Critical', 'High', 'Medium', 'Low']:
            count = len(df[df['priority'] == priority])
            if count > 0:
                print(f"      {priority.upper()}: {count}")
    
    return df

# Execute Step 1
repos_df = read_vulnerable_repos(
    status_filter="ready",
    priority_filter=None,  # None = all priorities
    limit=100
)
print("\n   ✅ Step 1 complete")

In [ ]:
# Preview the data
print("\n📋 Sample data (first 5 rows):")
repos_df.head()

---
## Step 2: Clone Repository

In [ ]:
def clone_repo(repo_url: str, clone_dir: str, repo_name: str) -> Optional[str]:
    """
    Clone a git repository to local directory.
    
    Returns: Path to cloned repo, or None if failed
    """
    print("\n" + "="*60)
    print("📦 STEP 2: Cloning Repository")
    print("="*60)
    print(f"   Repository: {repo_name}")
    print(f"   URL: {repo_url}")
    
    repo_path = Path(clone_dir) / repo_name
    
    # Check if already cloned
    if repo_path.exists():
        print(f"   ℹ️ Already cloned at: {repo_path}")
        # Pull latest
        print("   📥 Pulling latest changes...")
        result = subprocess.run(
            ["git", "pull"],
            cwd=str(repo_path),
            capture_output=True,
            text=True
        )
        if result.returncode == 0:
            print(f"   ✅ Updated successfully")
        else:
            print(f"   ⚠️ Pull failed (maybe detached HEAD): {result.stderr[:100]}")
        return str(repo_path)
    
    # Create clone directory
    Path(clone_dir).mkdir(parents=True, exist_ok=True)
    
    # Clone with depth 1 for speed
    print("   🔄 Cloning (shallow)...")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", repo_url, str(repo_path)],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print(f"   ✅ Cloned to: {repo_path}")
        
        # Count Java files
        java_files = list(repo_path.rglob("*.java"))
        print(f"   📄 Found {len(java_files)} Java files")
        
        return str(repo_path)
    else:
        print(f"   ❌ Clone failed: {result.stderr}")
        return None

In [ ]:
# Select a repo to process (pick the first one for demo)
# Filter for jackson-databind since we have good test data for it
jackson_repos = repos_df[repos_df['library'] == 'jackson-databind']
if len(jackson_repos) > 0:
    selected_row = jackson_repos.iloc[0]
else:
    selected_row = repos_df.iloc[0]

# Get candidate version (handle both list and string formats)
if isinstance(selected_row['candidate_versions'], list):
    target_version = selected_row['candidate_versions'][0]['version']
elif isinstance(selected_row['candidate_versions'], str):
    import json
    cv = json.loads(selected_row['candidate_versions'])
    target_version = cv[0]['version'] if cv else 'unknown'
else:
    target_version = 'unknown'

# Get CVE IDs (handle both list and string formats)
cve_ids = selected_row.get('cve_ids', [])
if isinstance(cve_ids, str):
    import json
    cve_ids = json.loads(cve_ids)
cve_display = ', '.join(cve_ids) if cve_ids else 'N/A'

print("🎯 Selected for processing:")
print(f"   Repo: {selected_row['repo_name']}")
print(f"   Library: {selected_row['library']}")
print(f"   Current version: {selected_row['current_version']}")
print(f"   Target version: {target_version}")
print(f"   CVE: {cve_display}")
print(f"   CVSS: {selected_row.get('cvss_score', 'N/A')}")

In [ ]:
# Execute Step 2
repo_path = clone_repo(
    repo_url=selected_row['repo_url'],
    clone_dir=CLONE_DIR,
    repo_name=selected_row['repo_name']
)

if repo_path:
    print("\n   ✅ Step 2 complete")
else:
    print("\n   ❌ Step 2 failed - cannot continue")

---
## Step 3: Get API Diff Between Library Versions

In [ ]:
def get_api_diff(
    library: str, 
    from_version: str, 
    to_version: str, 
    clone_dir: str,
    library_urls: Dict[str, str]
) -> Dict[str, Any]:
    """
    Get API changes between two versions of a library.
    
    Clones the library at both versions from GitHub and compares them.
    This is what LibraryMigrationAnalyzer does on Databricks.
    
    Returns dict with:
    - removed_methods: list of removed method signatures
    - changed_methods: list of methods with changed signatures
    - added_methods: list of new methods
    - breaking_changes: list of breaking changes
    """
    print("\n" + "="*60)
    print("🔍 STEP 3: Getting API Diff")
    print("="*60)
    print(f"   Library: {library}")
    print(f"   From: {from_version}")
    print(f"   To: {to_version}")
    
    result = {
        "library": library,
        "from_version": from_version,
        "to_version": to_version,
        "removed_methods": [],
        "changed_methods": [],
        "removed_classes": [],
        "added_methods": [],
        "breaking_changes": [],
        "source": None
    }
    
    # Get library GitHub URL
    library_url = library_urls.get(library)
    if not library_url:
        print(f"   ⚠️ No GitHub URL configured for {library}")
        print(f"   ℹ️ Add it to LIBRARY_GITHUB_URLS dict")
        result["source"] = "no_url"
        return result
    
    print(f"   📦 Library repo: {library_url}")
    
    # Clone library at both versions
    lib_clone_dir = Path(clone_dir) / "libraries"
    lib_clone_dir.mkdir(parents=True, exist_ok=True)
    
    v1_path = lib_clone_dir / f"{library}_v1"
    v2_path = lib_clone_dir / f"{library}_v2"
    
    def clone_at_version(url: str, dest: Path, version: str) -> bool:
        """Clone repo at specific version/tag."""
        if dest.exists():
            shutil.rmtree(dest)
        
        # Try common version tag formats
        version_tags = [
            version,                              # 2.5.2
            f"v{version}",                        # v2.5.2
            f"{library}-{version}",               # jackson-databind-2.5.2
            f"rel/{version}",                     # rel/2.5.2
        ]
        
        for tag in version_tags:
            try:
                cmd = ["git", "clone", "--depth", "1", "--branch", tag, url, str(dest)]
                subprocess.run(cmd, check=True, capture_output=True, text=True)
                print(f"      ✅ Cloned with tag: {tag}")
                return True
            except subprocess.CalledProcessError:
                continue
        
        return False
    
    # Clone v1 (current version)
    print(f"   📥 Cloning {library} @ {from_version}...")
    if not clone_at_version(library_url, v1_path, from_version):
        print(f"      ⚠️ Could not find tag for version {from_version}")
        result["source"] = "clone_failed"
        # Continue anyway - we can still find usages
    
    # Clone v2 (target version)
    print(f"   📥 Cloning {library} @ {to_version}...")
    if not clone_at_version(library_url, v2_path, to_version):
        print(f"      ⚠️ Could not find tag for version {to_version}")
        result["source"] = "clone_failed"
    
    # Compare the two versions if both cloned successfully
    if v1_path.exists() and v2_path.exists():
        result["source"] = "github_clone"
        
        # Extract public methods from both versions
        print("   🔬 Comparing API declarations...")
        
        v1_methods = extract_public_methods(v1_path)
        v2_methods = extract_public_methods(v2_path)
        
        print(f"      V1 public methods: {len(v1_methods)}")
        print(f"      V2 public methods: {len(v2_methods)}")
        
        # Find removed methods
        v1_signatures = set(v1_methods.keys())
        v2_signatures = set(v2_methods.keys())
        
        removed = v1_signatures - v2_signatures
        added = v2_signatures - v1_signatures
        
        for sig in removed:
            result["removed_methods"].append(sig)
            result["breaking_changes"].append({
                "type": "method_removed",
                "signature": sig,
                "class": v1_methods[sig].get("class", "Unknown")
            })
        
        for sig in added:
            result["added_methods"].append(sig)
        
        print(f"\n   📊 API Changes Summary:")
        print(f"      Removed methods: {len(result['removed_methods'])}")
        print(f"      Added methods: {len(result['added_methods'])}")
        print(f"      Breaking changes: {len(result['breaking_changes'])}")
    else:
        print("   ⚠️ Could not clone both versions - skipping API comparison")
        result["source"] = "partial"
    
    return result


def extract_public_methods(repo_path: Path) -> Dict[str, Dict]:
    """
    Extract public method signatures from Java source files.
    
    Returns dict: signature -> {class, file, line}
    """
    methods = {}
    
    # Find all Java files in src/main
    java_files = list(repo_path.rglob("src/main/**/*.java"))
    if not java_files:
        java_files = list(repo_path.rglob("*.java"))[:100]  # Limit for large repos
    
    for java_file in java_files[:200]:  # Limit to avoid long processing
        try:
            content = java_file.read_text(encoding='utf-8', errors='ignore')
            
            # Extract class name
            class_match = re.search(r'public\s+(?:abstract\s+)?class\s+(\w+)', content)
            class_name = class_match.group(1) if class_match else "Unknown"
            
            # Extract public method signatures (simple regex approach)
            method_pattern = r'public\s+(?:static\s+)?(?:final\s+)?(\w+(?:<[^>]+>)?)\s+(\w+)\s*\(([^)]*)\)'
            
            for match in re.finditer(method_pattern, content):
                return_type = match.group(1)
                method_name = match.group(2)
                params = match.group(3).strip()
                
                # Simplify params (remove variable names, keep types)
                param_types = []
                if params:
                    for param in params.split(','):
                        parts = param.strip().split()
                        if parts:
                            param_types.append(parts[0])  # Type only
                
                signature = f"{class_name}.{method_name}({', '.join(param_types)})"
                methods[signature] = {
                    "class": class_name,
                    "return_type": return_type,
                    "file": str(java_file.name)
                }
                
        except Exception:
            continue
    
    return methods

In [ ]:
# Execute Step 3
api_diff = get_api_diff(
    library=selected_row['library'],
    from_version=selected_row['current_version'],
    to_version=selected_row['candidate_versions'].split(',')[0].strip(),  # Take first candidate
    clone_dir=CLONE_DIR,
    library_urls=LIBRARY_GITHUB_URLS
)

print("\n   ✅ Step 3 complete")

In [ ]:
# Show some breaking changes
if api_diff["breaking_changes"]:
    print("\n📋 Sample breaking changes (first 10):")
    for i, change in enumerate(api_diff["breaking_changes"][:10]):
        print(f"   {i+1}. [{change['type']}] {change.get('class', '')}.{change['signature'][:60]}...")
elif api_diff["source"] == "no_url":
    print("\n   ⚠️ Add GitHub URL for this library to LIBRARY_GITHUB_URLS")
elif api_diff["source"] == "clone_failed":
    print("\n   ⚠️ Could not clone library versions - check tag format")
else:
    print("\n   ℹ️ No breaking changes detected between versions")

---
## Step 4: Find Library Usages in Source Code

In [ ]:
def find_library_usages(repo_path: str, library: str) -> Dict[str, Any]:
    """
    Find all usages of a library in the source code.
    
    Uses AST parsing to find:
    - Import statements
    - Class instantiations
    - Method calls
    
    Returns dict with file -> usages mapping
    """
    print("\n" + "="*60)
    print("🔎 STEP 4: Finding Library Usages")
    print("="*60)
    print(f"   Repository: {repo_path}")
    print(f"   Library: {library}")
    
    result = {
        "library": library,
        "files_scanned": 0,
        "files_with_usages": 0,
        "usages": [],
        "imports": [],
        "errors": []
    }
    
    # Map library name to package patterns
    library_packages = {
        "jackson-databind": ["com.fasterxml.jackson", "org.codehaus.jackson"],
        "log4j-core": ["org.apache.logging.log4j"],
        "spring-boot": ["org.springframework.boot"],
        "commons-collections": ["org.apache.commons.collections"],
        "snakeyaml": ["org.yaml.snakeyaml"],
    }
    
    # Get package patterns for this library
    patterns = library_packages.get(library, [library.replace("-", ".")])
    print(f"   📦 Looking for packages: {patterns}")
    
    # Scan Java files
    repo = Path(repo_path)
    java_files = list(repo.rglob("*.java"))
    result["files_scanned"] = len(java_files)
    print(f"   📄 Scanning {len(java_files)} Java files...")
    
    for java_file in java_files:
        try:
            content = java_file.read_text(encoding='utf-8', errors='ignore')
            
            # Quick check: does this file import the library?
            has_import = any(p in content for p in patterns)
            if not has_import:
                continue
            
            result["files_with_usages"] += 1
            rel_path = str(java_file.relative_to(repo))
            
            # Extract import statements
            import_lines = []
            for i, line in enumerate(content.split('\n'), 1):
                if line.strip().startswith('import ') and any(p in line for p in patterns):
                    import_lines.append({
                        "line_number": i,
                        "statement": line.strip()
                    })
                    result["imports"].append({
                        "file": rel_path,
                        "line": i,
                        "import": line.strip()
                    })
            
            # For detailed usage, we'd use JavaCodeTransformer
            # For now, extract simple patterns
            usages_in_file = []
            
            # Look for ObjectMapper (jackson), Logger (log4j), etc.
            common_classes = {
                "jackson-databind": ["ObjectMapper", "JsonNode", "JsonParser", "JsonGenerator"],
                "log4j-core": ["Logger", "LogManager"],
                "snakeyaml": ["Yaml", "DumperOptions"],
            }
            
            classes_to_find = common_classes.get(library, [])
            for i, line in enumerate(content.split('\n'), 1):
                for cls in classes_to_find:
                    if cls in line and not line.strip().startswith('//'):
                        usages_in_file.append({
                            "line_number": i,
                            "class": cls,
                            "code": line.strip()[:100]
                        })
            
            if usages_in_file:
                result["usages"].append({
                    "file": rel_path,
                    "imports": import_lines,
                    "usages": usages_in_file
                })
                
        except Exception as e:
            result["errors"].append({"file": str(java_file), "error": str(e)})
    
    print(f"\n   📊 Usage Summary:")
    print(f"      Files scanned: {result['files_scanned']}")
    print(f"      Files with usages: {result['files_with_usages']}")
    print(f"      Total imports: {len(result['imports'])}")
    print(f"      Files with detailed usages: {len(result['usages'])}")
    if result['errors']:
        print(f"      Errors: {len(result['errors'])}")
    
    return result

In [ ]:
# Execute Step 4
if repo_path:
    library_usages = find_library_usages(
        repo_path=repo_path,
        library=selected_row['library']
    )
    print("\n   ✅ Step 4 complete")
else:
    print("   ❌ Cannot scan - repo not cloned")

In [ ]:
# Show files with usages
if library_usages.get("usages"):
    print("\n📋 Files with library usages:")
    for usage in library_usages["usages"][:5]:
        print(f"\n   📄 {usage['file']}")
        print(f"      Imports: {len(usage['imports'])}")
        for imp in usage['imports'][:3]:
            print(f"         L{imp['line_number']}: {imp['statement']}")
        print(f"      Usages: {len(usage['usages'])}")
        for u in usage['usages'][:3]:
            print(f"         L{u['line_number']}: {u['class']} - {u['code'][:60]}...")
else:
    print("\n   ℹ️ No usages found (library might use different package names)")

---
## Step 5: Match Breaking Changes with Actual Usages

In [ ]:
def generate_code_fix_prompt(
    file_path: str,
    original_code: str,
    impacts: List[Dict],
    api_diff: Dict[str, Any],
    from_version: str,
    to_version: str
) -> str:
    """
    Generate a prompt for the LLM to fix the code.
    
    This is the ONLY step that requires an LLM!
    """
    prompt = f"""You are a Java migration expert. Fix the following code to be compatible with the library upgrade.

## Library Upgrade
- Library: {api_diff['library']}
- From: {from_version}
- To: {to_version}

## Breaking Changes That Affect This File
"""
    
    for impact in impacts:
        prompt += f"\n- Line {impact['line']}: Uses `{impact['api']}` which has changed"
    
    prompt += f"""

## API Changes
Removed methods:
"""
    for method in api_diff.get("removed_methods", [])[:10]:
        prompt += f"- {method}\n"
    
    prompt += f"""
## Original Code
File: {file_path}

```java
{original_code}
```

## Instructions
1. Update the code to be compatible with {api_diff['library']} {to_version}
2. Preserve the original functionality
3. Add comments explaining the migration changes
4. Return ONLY the fixed Java code, no explanations

## Fixed Code
```java
"""
    return prompt


def call_databricks_llm(prompt: str, endpoint: str = None, max_tokens: int = 4096) -> str:
    """
    Call Databricks Foundation Model API to generate code fix.
    
    Uses the serving endpoint configured in the workspace.
    """
    if not ON_DATABRICKS or w is None:
        return "// [LLM not available - running locally]\n// " + prompt[:200] + "..."
    
    endpoint = endpoint or LLM_ENDPOINT
    
    try:
        response = w.serving_endpoints.query(
            name=endpoint,
            messages=[
                ChatMessage(
                    role=ChatMessageRole.SYSTEM,
                    content="You are a Java migration expert. Generate only valid Java code, no explanations."
                ),
                ChatMessage(
                    role=ChatMessageRole.USER,
                    content=prompt
                )
            ],
            max_tokens=max_tokens,
            temperature=0.1  # Low temperature for deterministic code generation
        )
        
        return response.choices[0].message.content
        
    except Exception as e:
        return f"// [LLM Error: {str(e)}]"


def generate_code_fix(
    required_changes: List[Dict],
    repo_path: str,
    api_diff: Dict[str, Any],
    from_version: str,
    to_version: str,
    call_llm: bool = True
) -> List[Dict]:
    """
    Generate code fixes for all affected files.
    
    This is the ONLY step that needs an LLM!
    
    Args:
        call_llm: If True, actually call the LLM. If False, just show the prompt.
    """
    print("\n" + "="*60)
    print("🤖 STEP 6: Generating Code Fixes with LLM")
    print("="*60)
    print(f"   ⚡ This is the ONLY step that requires an LLM!")
    print(f"   📁 Files to fix: {len(required_changes)}")
    print(f"   🤖 LLM endpoint: {LLM_ENDPOINT}")
    print(f"   🔧 Call LLM: {call_llm}")
    
    if not ON_DATABRICKS and call_llm:
        print("   ⚠️ LLM calls require Databricks - showing prompts only")
        call_llm = False
    
    fixes = []
    
    for change in required_changes[:3]:  # Limit to 3 for demo
        file_path = change["file"]
        impacts = change["impacts"]
        
        print(f"\n   Processing: {file_path}")
        
        # Read the original file
        try:
            full_path = Path(repo_path) / file_path
            original_code = full_path.read_text(encoding='utf-8', errors='ignore')
        except Exception as e:
            print(f"      ❌ Could not read file: {e}")
            continue
        
        # Generate prompt
        prompt = generate_code_fix_prompt(
            file_path=file_path,
            original_code=original_code[:3000],  # Limit size
            impacts=impacts,
            api_diff=api_diff,
            from_version=from_version,
            to_version=to_version
        )
        
        if call_llm:
            print(f"      🤖 Calling LLM ({len(prompt)} chars)...")
            fixed_code = call_databricks_llm(prompt)
            print(f"      ✅ Received {len(fixed_code)} chars")
        else:
            print(f"      📝 Generated prompt ({len(prompt)} chars)")
            fixed_code = None
        
        fixes.append({
            "file": file_path,
            "original_lines": len(original_code.split('\n')),
            "impacts": len(impacts),
            "prompt_length": len(prompt),
            "fixed_code": fixed_code,
            "prompt": prompt[:500] + "..." if len(prompt) > 500 else prompt
        })
    
    print(f"\n   📊 Fix Generation Summary:")
    print(f"      Files processed: {len(fixes)}")
    print(f"      Total prompt chars: {sum(f['prompt_length'] for f in fixes)}")
    if call_llm:
        print(f"      LLM responses: {sum(1 for f in fixes if f['fixed_code'])}")
    
    return fixes

In [ ]:
# Execute Step 5
impact_analysis = match_changes_with_usages(api_diff, library_usages)
print("\n   ✅ Step 5 complete")

In [ ]:
# Show required changes
if impact_analysis["required_changes"]:
    print("\n📋 Required code changes:")
    for change in impact_analysis["required_changes"][:5]:
        print(f"\n   📄 {change['file']}")
        for impact in change["impacts"][:3]:
            print(f"      L{impact['line']}: Uses '{impact['api']}'")
            print(f"         {impact['code'][:70]}...")
else:
    print("\n   ℹ️ No breaking changes affect the codebase!")
    print("   ✅ The library can be upgraded safely.")

---
## Step 6: Generate Code Fix with LLM (The ONLY LLM Step!)

In [ ]:
def write_results_to_uc(
    repo_name: str,
    library: str,
    repo_id: str,
    target_version: str,
    impact_analysis: Dict,
    code_fixes: List[Dict],
) -> Dict:
    """
    Write analysis results back to Unity Catalog.
    
    Uses UCIntegration service which auto-detects DATA_MODE:
    - "mock" → logs what would be written (no persistence)
    - "production" → writes to Unity Catalog tables
    """
    print("\n" + "="*60)
    print("💾 STEP 7: Writing Results to Unity Catalog")
    print("="*60)
    print(f"   Repository: {repo_name}")
    print(f"   Library: {library}")
    print(f"   Mode: {DATA_MODE}")
    
    # Calculate metrics
    files_impacted = impact_analysis["summary"].get("affected_file_count", 0)
    total_impacts = impact_analysis["summary"].get("total_impacts", 0)
    fixes_generated = len([f for f in code_fixes if f.get("fixed_code")])
    
    # Determine confidence based on fix generation
    if fixes_generated > 0:
        confidence = 0.85
    elif files_impacted == 0:
        confidence = 0.95  # Safe upgrade
    else:
        confidence = 0.5  # Changes needed but no fixes yet
    
    try:
        from services.uc_integration import UCIntegration
        
        uc = UCIntegration(catalog=UC_CATALOG, schema=UC_SCHEMA)
        
        # Write analysis result
        analysis_id = uc.write_analysis_result(
            repo_id=repo_id,
            target_version=target_version,
            code_changes_files=files_impacted,
            code_changes_callsites=total_impacts,
            confidence=confidence,
            breaking_changes=impact_analysis.get("affected_files", []),
            notes=f"Status: {impact_analysis['summary'].get('status', 'unknown')}"
        )
        
        # Update repo status if we analyzed it
        if DATA_MODE == "production":
            uc.update_repo_status(repo_id, "analyzed")
        
        print(f"\n   ✅ Analysis saved")
        print(f"      analysis_id: {analysis_id}")
        
    except ImportError:
        print("   ⚠️ UCIntegration not available - skipping write")
        analysis_id = "mock-" + repo_id
    
    # Prepare result summary
    result = {
        "repo_name": repo_name,
        "library": library,
        "analysis_id": analysis_id,
        "analysis_timestamp": pd.Timestamp.now().isoformat(),
        "status": impact_analysis["summary"].get("status", "unknown"),
        "affected_files": files_impacted,
        "total_impacts": total_impacts,
        "fixes_generated": fixes_generated,
        "confidence": confidence,
    }
    
    print("\n   📋 Result summary:")
    for key, value in result.items():
        print(f"      {key}: {value}")
    
    return result

In [ ]:
# Execute Step 6
if impact_analysis["required_changes"]:
    code_fixes = generate_code_fix(
        required_changes=impact_analysis["required_changes"],
        repo_path=repo_path,
        api_diff=api_diff,
        from_version=selected_row['current_version'],
        to_version=selected_row['candidate_versions'].split(',')[0].strip(),
        call_llm=ON_DATABRICKS  # Only call LLM on Databricks
    )
    print("\n   ✅ Step 6 complete")
else:
    code_fixes = []
    print("\n   ℹ️ No code fixes needed - upgrade is safe!")

In [ ]:
# Show a sample prompt
if code_fixes:
    print("\n📋 Sample LLM Prompt (first file):")
    print("="*60)
    print(code_fixes[0]["prompt"])
    print("="*60)
    
    if code_fixes[0].get("fixed_code"):
        print("\n📋 LLM Generated Fix:")
        print("="*60)
        print(code_fixes[0]["fixed_code"][:1000])
        print("="*60)

In [ ]:
# Show a sample prompt
if code_fixes:
    print("\n📋 Sample LLM Prompt (first file):")
    print("="*60)
    print(code_fixes[0]["prompt"])
    print("="*60)

---
## Step 7: Write Results to Unity Catalog

In [ ]:
def write_results_to_uc(
    repo_name: str,
    library: str,
    impact_analysis: Dict,
    code_fixes: List[Dict],
    use_uc: bool = False
) -> Dict:
    """
    Write analysis results back to Unity Catalog.
    
    In production, this writes to UC tables.
    For demo, just prints what would be written.
    """
    print("\n" + "="*60)
    print("💾 STEP 7: Writing Results to UC")
    print("="*60)
    print(f"   Repository: {repo_name}")
    print(f"   Library: {library}")
    print(f"   Use UC: {use_uc}")
    
    # Prepare the result record
    result = {
        "repo_name": repo_name,
        "library": library,
        "analysis_timestamp": pd.Timestamp.now().isoformat(),
        "status": impact_analysis["summary"].get("status", "unknown"),
        "affected_files": len(impact_analysis.get("affected_files", [])),
        "total_impacts": impact_analysis["summary"].get("total_impacts", 0),
        "fixes_generated": len(code_fixes),
        "details": json.dumps({
            "affected_files": impact_analysis.get("affected_files", []),
            "required_changes": impact_analysis.get("required_changes", [])
        })
    }
    
    if use_uc:
        # This would run on Databricks
        # df = spark.createDataFrame([result])
        # df.write.mode("append").saveAsTable("catalog.schema.migration_analysis")
        print("   📤 Would write to: catalog.schema.migration_analysis")
    else:
        print("   ℹ️ UC disabled - showing what would be written:")
    
    print("\n   📋 Result record:")
    for key, value in result.items():
        if key == "details":
            print(f"      {key}: <JSON with {len(value)} chars>")
        else:
            print(f"      {key}: {value}")
    
    return result

In [ ]:
# Execute Step 7
uc_result = write_results_to_uc(
    repo_name=selected_row['repo_name'],
    library=selected_row['library'],
    repo_id=selected_row.get('repo_id', f"{selected_row['repo_name']}_{selected_row['library']}"),
    target_version=selected_row['candidate_versions'][0]['version'] if isinstance(selected_row['candidate_versions'], list) else selected_row['candidate_versions'].split(',')[0].strip(),
    impact_analysis=impact_analysis,
    code_fixes=code_fixes,
)

print("\n   ✅ Step 7 complete")

---
## 📊 Pipeline Summary

In [ ]:
print("\n" + "="*60)
print("📊 PIPELINE EXECUTION SUMMARY")
print("="*60)

env_status = "🔷 Databricks" if ON_DATABRICKS else "💻 Local"
data_source = "Unity Catalog" if DATA_MODE == "production" else "CSV (mock)"

# Get target version safely
if isinstance(selected_row['candidate_versions'], list):
    target_ver = selected_row['candidate_versions'][0]['version']
else:
    target_ver = selected_row['candidate_versions'].split(',')[0].strip()

print(f"""
🌍 Environment: {env_status}
📊 Data Source: {data_source}

🎯 Target:
   Repository: {selected_row['repo_name']}
   Library: {selected_row['library']}
   Upgrade: {selected_row['current_version']} → {target_ver}
   CVE: {selected_row.get('cve_ids', 'N/A')}
   CVSS Score: {selected_row.get('cvss_score', 'N/A')}

📋 Steps Completed:
   ✅ Step 1: Read {len(repos_df)} repo/library combinations from {data_source}
   ✅ Step 2: Cloned repository to {repo_path or 'N/A'}
   ✅ Step 3: Got API diff ({len(api_diff.get('breaking_changes', []))} breaking changes)
   ✅ Step 4: Found usages in {library_usages.get('files_with_usages', 0)} files
   ✅ Step 5: Matched impacts ({impact_analysis['summary'].get('affected_file_count', 0)} affected files)
   ✅ Step 6: {"Generated" if DATA_MODE == "production" else "Prepared"} {len(code_fixes)} code fix{"es" if len(code_fixes) != 1 else ""}
   ✅ Step 7: {"Wrote to UC" if DATA_MODE == "production" else "Logged (mock mode)"}

🤖 LLM Usage:
   Endpoint: {LLM_ENDPOINT}
   Only Step 6 requires an LLM!
   All other steps are 100% deterministic.
   {"✅ LLM called for code generation" if DATA_MODE == "production" and code_fixes else "⚠️ LLM not called (mock mode or no changes)"}

📈 Result:
   Status: {impact_analysis['summary'].get('status', 'unknown')}
   Affected files: {impact_analysis['summary'].get('affected_file_count', 0)}
   Total code impacts: {impact_analysis['summary'].get('total_impacts', 0)}
""")

if impact_analysis['summary'].get('status') == 'safe_upgrade':
    print("   ✅ SAFE TO UPGRADE - No breaking changes affect this codebase!")
elif impact_analysis['summary'].get('status') == 'changes_required':
    if DATA_MODE == "production":
        print("   ✅ CODE FIXES GENERATED - Review and apply them")
    else:
        print("   ⚠️ CHANGES REQUIRED - Switch to production mode for LLM fixes")
else:
    print("   ℹ️ Analysis complete - review results above")
    
print(f"\n💡 To switch modes, change DATA_MODE at the top of the notebook:")
print(f"   Current: DATA_MODE = \"{DATA_MODE}\"")

---
## 🔄 Process All Repos (Batch Mode)

In [ ]:
def process_all_repos(
    repos_df: pd.DataFrame,
    clone_dir: str,
    library_urls: Dict[str, str],
    use_llm: bool = False,
    max_repos: int = 5
) -> pd.DataFrame:
    """
    Process all repositories in batch mode.
    
    Returns a DataFrame with results for each repo/library.
    """
    import json
    
    print("\n" + "="*60)
    print("🔄 BATCH PROCESSING MODE")
    print("="*60)
    print(f"   Total repos to process: {min(len(repos_df), max_repos)}")
    print(f"   LLM enabled: {use_llm}")
    
    results = []
    
    for idx, row in repos_df.head(max_repos).iterrows():
        print(f"\n{'='*60}")
        print(f"Processing [{idx+1}/{min(len(repos_df), max_repos)}]: {row['repo_name']} / {row['library']}")
        print("="*60)
        
        try:
            # Parse candidate_versions (handle list, string, or JSON string)
            cv = row.get('candidate_versions', [])
            if isinstance(cv, str):
                cv = json.loads(cv)
            if isinstance(cv, list) and len(cv) > 0:
                target_version = cv[0]['version'] if isinstance(cv[0], dict) else cv[0]
            else:
                target_version = 'unknown'
            
            # Parse CVE IDs (handle cve_ids list or cve_id string)
            cve_ids = row.get('cve_ids', row.get('cve_id', []))
            if isinstance(cve_ids, str):
                try:
                    cve_ids = json.loads(cve_ids)
                except json.JSONDecodeError:
                    cve_ids = [cve_ids] if cve_ids else []
            cve_display = ', '.join(cve_ids) if cve_ids else 'N/A'
            
            # Step 2: Clone
            repo_path = clone_repo(row['repo_url'], clone_dir, row['repo_name'])
            
            if not repo_path:
                results.append({
                    "repo_name": row['repo_name'],
                    "library": row['library'],
                    "status": "clone_failed",
                    "affected_files": 0,
                    "error": "Failed to clone repository"
                })
                continue
            
            # Step 3: Get API diff (clones from GitHub)
            api_diff = get_api_diff(
                library=row['library'],
                from_version=row['current_version'],
                to_version=target_version,
                clone_dir=clone_dir,
                library_urls=library_urls
            )
            
            # Step 4: Find usages
            library_usages = find_library_usages(repo_path, row['library'])
            
            # Step 5: Match
            impact = match_changes_with_usages(api_diff, library_usages)
            
            results.append({
                "repo_name": row['repo_name'],
                "library": row['library'],
                "current_version": row['current_version'],
                "target_version": target_version,
                "cve_ids": cve_display,
                "cvss_score": row.get('cvss_score', 'N/A'),
                "status": impact['summary'].get('status', 'unknown'),
                "affected_files": impact['summary'].get('affected_file_count', 0),
                "total_impacts": impact['summary'].get('total_impacts', 0),
                "files_with_usages": library_usages.get('files_with_usages', 0),
                "breaking_changes": len(api_diff.get('breaking_changes', []))
            })
            
        except Exception as e:
            results.append({
                "repo_name": row['repo_name'],
                "library": row['library'],
                "status": "error",
                "affected_files": 0,
                "error": str(e)
            })
    
    return pd.DataFrame(results)

In [ ]:
# Uncomment to run batch processing
# batch_results = process_all_repos(
#     repos_df=repos_df,
#     clone_dir=CLONE_DIR,
#     library_urls=LIBRARY_GITHUB_URLS,
#     use_llm=False,
#     max_repos=3
# )
# batch_results

print("\n💡 To run batch processing, uncomment the cell above.")